In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor

In [17]:
df = pd.read_csv('uber.csv')
df = df.drop(['Unnamed:0.1', 'Unnamed:0'], axis=1, errors='ignore')
df

,Unnamed: 0.1,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5
...,...,...,...,...,...,...,...,...,...,...
199995,199995,42598914,2012-10-28 10:49:00.00000053,3.0,2012-10-28 10:49:00 UTC,-73.987042,40.739367,-73.986525,40.740297,1
199996,199996,16382965,2014-03-14 01:09:00.0000008,7.5,2014-03-14 01:09:00 UTC,-73.984722,40.736837,-74.006672,40.739620,1
199997,199997,27804658,2009-06-29 00:42:00.00000078,30.9,2009-06-29 00:42:00 UTC,-73.986017,40.756487,-73.858957,40.692588,2
199998,199998,20259894,2015-05-20 14:56:25.0000004,14.5,2015-05-20 14:56:25 UTC,-73.997124,40.725452,-73.983215,40.695415,1


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0.1       200000 non-null  int64  
 1   Unnamed: 0         200000 non-null  int64  
 2   key                200000 non-null  str    
 3   fare_amount        200000 non-null  float64
 4   pickup_datetime    200000 non-null  str    
 5   pickup_longitude   200000 non-null  float64
 6   pickup_latitude    200000 non-null  float64
 7   dropoff_longitude  199999 non-null  float64
 8   dropoff_latitude   199999 non-null  float64
 9   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(3), str(2)
memory usage: 15.3 MB


In [19]:
numberic_cols = ['fare_amount', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']

# for col in numberic_cols:
#     df[col] = pd.to_numeric(df[col],errors='coerce')



In [20]:
df = df.dropna()


In [21]:
df = df.dropna(subset=numberic_cols + ['pickup_datetime'])
df = df[
    df['pickup_latitude'].between(-90, 90) &
    df['dropoff_latitude'].between(-90, 90) &
    df['pickup_longitude'].between(-180, 180) &
    df['dropoff_longitude'].between(-180, 180) &
    (df['fare_amount'] > 0) &
    (df['passenger_count'].between(1, 6))
]

df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce', utc=True)

df = df.dropna(subset=['pickup_datetime'])


In [22]:
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek
df

/tmp/ipykernel_33730/3272215496.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['dayofweek'] = df['pickup_datetime'].dt.dayofweek


,Unnamed: 0.1,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,hour,day,month,year,dayofweek
0,0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1,19,7,5,2015,3
1,1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,-73.994355,40.728225,-73.994710,40.750325,1,20,17,7,2009,4
2,2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,-74.005043,40.740770,-73.962565,40.772647,1,21,24,8,2009,0
3,3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,-73.976124,40.790844,-73.965316,40.803349,3,8,26,6,2009,4
4,4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,-73.925023,40.744085,-73.973082,40.761247,5,17,28,8,2014,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,199995,42598914,2012-10-28 10:49:00.00000053,3.0,2012-10-28 10:49:00+00:00,-73.987042,40.739367,-73.986525,40.740297,1,10,28,10,2012,6
199996,199996,16382965,2014-03-14 01:09:00.0000008,7.5,2014-03-14 01:09:00+00:00,-73.984722,40.736837,-74.006672,40.739620,1,1,14,3,2014,4
199997,199997,27804658,2009-06-29 00:42:00.00000078,30.9,2009-06-29 00:42:00+00:00,-73.986017,40.756487,-73.858957,40.692588,2,0,29,6,2009,0
199998,199998,20259894,2015-05-20 14:56:25.0000004,14.5,2015-05-20 14:56:25+00:00,-73.997124,40.725452,-73.983215,40.695415,1,14,20,5,2015,2


In [23]:
lat1 = np.radians(df['pickup_latitude'])
lat2 = np.radians(df['dropoff_latitude'])
lon1 = np.radians(df['pickup_longitude'])
lon2 = np.radians(df['dropoff_longitude'])

In [24]:
dlat = lat2 - lat1
dlon = lon2 - lon1


a = (
    np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

In [25]:
a = np.clip(a, 0, 1)
df['dist_travel_km'] = 6371 * 2 * np.arcsin(np.sqrt(a))
df = df.sample(
    n=min(50000, len(df)),
    random_state=42
).reset_index(drop=True)

print("Dataset size:", len(df))


Dataset size: 50000


In [26]:
features = [
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
    'passenger_count', 'hour', 'day', 'month',
    'year', 'dayofweek', 'dist_travel_km'
]

X = df[features]
Y = df['fare_amount']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [27]:
linear_model = LinearRegression()
linear_model.fit(X=x_train, y=y_train)
linear_model_prediction = linear_model.predict(x_test)

r2 = metrics.r2_score(y_true=y_test, y_pred=linear_model_prediction)
rmse = np.sqrt(metrics.mean_squared_error(y_test, linear_model_prediction))

print(f'R² score for Linear Regression: {r2:.4f}')
print(f'RMSE for Linear Regression: {rmse:.4f}')

R² score for Linear Regression: 0.0163
RMSE for Linear Regression: 9.6916


In [28]:
random_forest_model = RandomForestRegressor()

random_forest_model.fit(X=x_train, y=y_train)
random_forest_model_prediction = random_forest_model.predict(X=x_test)

r2 = metrics.r2_score(y_true=y_test, y_pred=random_forest_model_prediction)
rmse = np.sqrt(metrics.mean_squared_error(y_test, random_forest_model_prediction))

print(f'R² score for Random Forest Regression: {r2:.4f}')
print(f'RMSE for Random Forest Regression: {rmse:.4f}')

R² score for Random Forest Regression: 0.7913
RMSE for Random Forest Regression: 4.4636


# Simplest Version

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load dataset
df = pd.read_csv("uber.csv", on_bad_lines="skip")

# 2. Preprocessing
df = df.dropna()
df = df.drop_duplicates()
df = df[df["fare_amount"] > 0]

# 3. Extract date and time features
df["pickup_datetime"] = pd.to_datetime(
    df["pickup_datetime"], errors="coerce", utc=True
)
df = df.dropna(subset=["pickup_datetime"])

df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["dayofweek"] = df["pickup_datetime"].dt.dayofweek

# 4. Calculate distance
lat1 = np.radians(df["pickup_latitude"])
lon1 = np.radians(df["pickup_longitude"])
lat2 = np.radians(df["dropoff_latitude"])
lon2 = np.radians(df["dropoff_longitude"])

a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin((lon2 - lon1) / 2) ** 2
)

df["distance"] = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# 5. Remove invalid distances
df = df[df["distance"].between(1, 130)].copy()

# 6. Select features and target
features = [
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "passenger_count", "hour", "day",
    "month", "year", "dayofweek", "distance"
]

X = df[features]
y = df["fare_amount"]

# Use a sample for faster training
if len(df) > 50000:
    X = X.sample(50000, random_state=42)
    y = y.loc[X.index]

# 7. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 8. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("Linear Regression")
print("R2:", r2_score(y_test, pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_lr)))

# 9. Random Forest Regression
rf = RandomForestRegressor(
    n_estimators=30, max_depth=15, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("\nRandom Forest Regression")
print("R2:", r2_score(y_test, pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_rf)))
```